# Round11 Full Workflow Notebook: `v11_4_guard_ec.csv`\n\n**策略标签**：Guarded EC + TA balanced branch\n\n本 notebook 是完整流程版本：从原始数据导入到最终提交文件导出与分析。

## Step 1. Problem Framing

**这个步骤做什么**
明确本分支目标与风险等级。

**为什么要做这个步骤**
先定义策略目标，避免盲目调参。

**预期产出**
得到本分支的实验定位。

## Step 2. Environment Setup

**这个步骤做什么**
导入依赖并初始化运行环境。

**为什么要做这个步骤**
保证流程可复现。

**预期产出**
得到统一环境与路径常量。

In [ ]:
# =========================
# Step: Environment Setup
# 作用：导入依赖库，设置显示选项，定义路径和 GitHub 数据地址
# =========================

# 基础数值与表格库
import numpy as np
import pandas as pd

# 可视化（用于诊断分布）
import matplotlib.pyplot as plt

# 路径管理
from pathlib import Path

# 模型与工具
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.neighbors import NearestNeighbors

# 显示设置：便于在 notebook 中阅读表格
pd.set_option('display.max_columns', 220)
pd.set_option('display.width', 160)

# 项目路径（主要用于导出文件）
PROJECT_ROOT = Path('/Users/zhangziling/Documents/New project')

# 官方 GitHub 原始数据地址（原始数据全部从这里读取）
BASE_RAW = 'https://raw.githubusercontent.com/Snowflake-Labs/EY-AI-and-Data-Challenge/refs/heads/main'

## Step 3. Load Raw Data

**这个步骤做什么**
从 GitHub 读取官方原始数据。

**为什么要做这个步骤**
保证分析起点一致，来源可追溯。

**预期产出**
得到 6 张原始输入表。

In [ ]:
# =========================
# Step: Load Official Raw Datasets
# 作用：从 GitHub 读取训练/验证原始数据
# =========================

# 训练数据：目标水质 + Landsat + TerraClimate
train_wq = pd.read_csv(f'{BASE_RAW}/water_quality_training_dataset.csv')
train_ls = pd.read_csv(f'{BASE_RAW}/landsat_features_training.csv')
train_tc = pd.read_csv(f'{BASE_RAW}/terraclimate_features_training.csv')

# 验证数据：submission_template + Landsat + TerraClimate
val_tpl = pd.read_csv(f'{BASE_RAW}/submission_template.csv')
val_ls = pd.read_csv(f'{BASE_RAW}/landsat_features_validation.csv')
val_tc = pd.read_csv(f'{BASE_RAW}/terraclimate_features_validation.csv')

print('train_wq:', train_wq.shape)
print('train_ls:', train_ls.shape)
print('train_tc:', train_tc.shape)
print('val_tpl:', val_tpl.shape)
print('val_ls:', val_ls.shape)
print('val_tc:', val_tc.shape)

## Step 4. Data QA

**这个步骤做什么**
检查结构和缺失率。

**为什么要做这个步骤**
提前识别异常，减少后续错误。

**预期产出**
得到基础质量报告。

In [ ]:
# =========================
# Step: Quick Data QA
# 作用：检查数据结构、缺失率、主键完整性
# =========================

def quick_check(df: pd.DataFrame, name: str, topn: int = 8):
    # 打印基础结构
    print(f'\n{name}')
    print('shape:', df.shape)
    print('columns:', list(df.columns))

    # 打印缺失率最高字段
    miss = df.isna().mean().sort_values(ascending=False).head(topn)
    print('top missing ratio:')
    print(miss)

quick_check(train_wq, 'train_wq')
quick_check(train_ls, 'train_ls')
quick_check(train_tc, 'train_tc')
quick_check(val_tpl, 'val_tpl')

## Step 5. Merge Multi-source Data

**这个步骤做什么**
按主键融合多源表。

**为什么要做这个步骤**
构建统一训练/验证主表。

**预期产出**
得到 train_full / val_full。

In [ ]:
# =========================
# Step: Merge Multi-source Tables
# 作用：按统一主键把多源特征拼接成训练主表和验证主表
# =========================

KEY = ['Latitude', 'Longitude', 'Sample Date']

# 日期统一成 datetime，避免 merge 键类型不一致
for df in [train_wq, train_ls, train_tc, val_tpl, val_ls, val_tc]:
    df['Sample Date'] = pd.to_datetime(df['Sample Date'], dayfirst=True, errors='coerce')

# 训练主表
train_full = (
    train_wq
    .merge(train_ls, on=KEY, how='left', suffixes=('', '_ls'))
    .merge(train_tc, on=KEY, how='left', suffixes=('', '_tc'))
    .copy()
)

# 验证主表
val_full = (
    val_tpl
    .merge(val_ls, on=KEY, how='left', suffixes=('', '_ls'))
    .merge(val_tc, on=KEY, how='left', suffixes=('', '_tc'))
    .copy()
)

print('train_full:', train_full.shape)
print('val_full:', val_full.shape)
print('train key unique:', train_full[KEY].drop_duplicates().shape[0], '/', len(train_full))
print('val key unique:', val_full[KEY].drop_duplicates().shape[0], '/', len(val_full))

## Step 6. Feature Engineering

**这个步骤做什么**
构造时间/空间/光谱特征。

**为什么要做这个步骤**
提升模型表达能力与解释性。

**预期产出**
得到 train_feat / val_feat。

In [ ]:
# =========================
# Step: Feature Engineering
# 作用：构造时间/空间/光谱组合特征
# =========================

EPS = 1e-6

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # ---- 时间特征 ----
    dt = out['Sample Date']
    out['year'] = dt.dt.year.astype(float)
    out['month'] = dt.dt.month.astype(float)
    out['dayofyear'] = dt.dt.dayofyear.astype(float)
    out['month_sin'] = np.sin(2 * np.pi * out['month'] / 12)
    out['month_cos'] = np.cos(2 * np.pi * out['month'] / 12)
    out['doy_sin'] = np.sin(2 * np.pi * out['dayofyear'] / 366)
    out['doy_cos'] = np.cos(2 * np.pi * out['dayofyear'] / 366)

    # ---- 空间特征 ----
    out['lat_abs'] = out['Latitude'].abs()
    out['lon_abs'] = out['Longitude'].abs()
    out['lat_x_lon'] = out['Latitude'] * out['Longitude']

    # ---- 光谱与水文组合 ----
    out['ndvi'] = (out['nir'] - out['green']) / (out['nir'] + out['green'] + EPS)
    out['ndbi'] = (out['swir16'] - out['nir']) / (out['swir16'] + out['nir'] + EPS)
    out['swir_ratio'] = out['swir22'] / (out['swir16'] + EPS)
    out['pet_x_ndmi'] = out['pet'] * out['NDMI']
    out['pet_x_mndwi'] = out['pet'] * out['MNDWI']

    return out

train_feat = add_features(train_full)
val_feat = add_features(val_full)
print('train_feat:', train_feat.shape)
print('val_feat:', val_feat.shape)

## Step 7. Load Proven Anchor

**这个步骤做什么**
加载上一轮线上最优提交作为锚点。

**为什么要做这个步骤**
保证策略有稳定基线，降低回撤风险。

**预期产出**
得到 v10_best 和 v2_ref。

In [ ]:
# =========================
# Step: Load Proven Online Anchor
# 作用：加载上一轮线上最优提交（v10_4）作为稳定锚点
# =========================

# 说明：这不是原始数据集，而是上一轮已验证的预测锚点。
v10_best = pd.read_csv(PROJECT_ROOT / 'v10_4_ec_ta_link_w30_near.csv')
v2_ref = pd.read_csv(PROJECT_ROOT / 'submission_template_v2.csv')

print('v10_best:', v10_best.shape)
print('v2_ref:', v2_ref.shape)

## Step 8. Model Helper Functions

**这个步骤做什么**
定义训练与融合函数。

**为什么要做这个步骤**
避免重复代码并保持逻辑一致。

**预期产出**
得到建模与导出工具函数。

In [ ]:
# =========================
# Step: Modeling Helper Functions
# 作用：封装训练、预测、融合、距离衰减函数
# =========================

def fit_predict_et(train_df, val_df, features, target, transform='none', n_estimators=900):
    # 1) 取特征和标签
    Xtr = train_df[features]
    Xva = val_df[features]
    y = train_df[target].to_numpy(dtype=float)

    # 2) 目标变换（处理偏态）
    if transform == 'log1p':
        y_mod = np.log1p(np.clip(y, 0, None))
    else:
        y_mod = y

    # 3) 缺失填补
    imp = SimpleImputer(strategy='median')
    Xtr_i = imp.fit_transform(Xtr)
    Xva_i = imp.transform(Xva)

    # 4) 训练 ET 模型
    model = ExtraTreesRegressor(
        n_estimators=n_estimators,
        max_depth=None,
        min_samples_leaf=2,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1,
    )
    model.fit(Xtr_i, y_mod)
    pred = model.predict(Xva_i)

    # 5) 逆变换
    if transform == 'log1p':
        pred = np.expm1(pred)

    return np.clip(pred, 0, None)


def distance_scale(train_coords, val_coords):
    # 最近邻距离归一化：距离越大，说明 OOD 风险越高
    tr_unique = train_coords[['Latitude', 'Longitude']].drop_duplicates().to_numpy()
    va = val_coords[['Latitude', 'Longitude']].to_numpy()

    nn = NearestNeighbors(n_neighbors=1, metric='euclidean')
    nn.fit(tr_unique)
    dist, _ = nn.kneighbors(va)
    dist = dist.reshape(-1)

    q10, q90 = np.quantile(dist, [0.10, 0.90])
    return np.clip((dist - q10) / max(q90 - q10, 1e-9), 0.0, 1.0)


def ec_tail_guard(ec, ec_base, scale, strong=False):
    # EC 风险保护：限制极端尾部 + 远距离点向基线回退
    out = ec.copy()
    if strong:
        qlo, qhi = np.quantile(out, [0.04, 0.96])
        blend0, extra, start = 0.10, 0.12, 0.55
    else:
        qlo, qhi = np.quantile(out, [0.03, 0.97])
        blend0, extra, start = 0.06, 0.08, 0.65

    out = np.clip(out, qlo, qhi)
    out = (1 - blend0) * out + blend0 * ec_base
    extra_w = extra * np.clip((scale - start) / max(1.0 - start, 1e-9), 0, 1)
    out = (1 - extra_w) * out + extra_w * ec_base
    return np.clip(out, 0, None)


def blend_with_scale(base, model, max_w, scale, power, near_boost):
    # 距离衰减融合：近点更相信模型，远点更保守
    w = max_w * (1 - np.power(scale, power))
    w = w * (1 + near_boost * (1 - scale))
    w = np.clip(w, 0, 0.80)
    return np.clip(base + w * (model - base), 0, None)


def build_submission(val_template_df, ta, ec, drp):
    # 统一输出列结构
    sub = val_template_df[['Latitude', 'Longitude', 'Sample Date']].copy()
    sub['Sample Date'] = pd.to_datetime(sub['Sample Date']).dt.strftime('%d/%m/%Y')
    sub['Total Alkalinity'] = np.clip(ta, 0, None)
    sub['Electrical Conductance'] = np.clip(ec, 0, None)
    sub['Dissolved Reactive Phosphorus'] = np.clip(drp, 0, None)
    return sub[['Latitude','Longitude','Sample Date','Total Alkalinity','Electrical Conductance','Dissolved Reactive Phosphorus']]

## Step 9. Build Core Components

**这个步骤做什么**
训练 TA/DRP 组件并计算 OOD scale。

**为什么要做这个步骤**
为策略渲染提供可复用预测组件。

**预期产出**
得到 ta_base/ta_link/drp_base/drp_link/ec_base_val。

In [ ]:
# =========================
# Step: Build Core Components
# 作用：训练 TA/DRP 基础模型与 EC联动模型组件
# =========================

# 基础特征列（训练和验证都存在）
candidate_features = [
    'nir','green','swir16','swir22','NDMI','MNDWI','pet',
    'Latitude','Longitude','year','month','dayofyear',
    'month_sin','month_cos','doy_sin','doy_cos',
    'lat_abs','lon_abs','lat_x_lon',
    'ndvi','ndbi','swir_ratio','pet_x_ndmi','pet_x_mndwi'
]
base_features = [c for c in candidate_features if c in train_feat.columns and c in val_feat.columns]

# 锚点数组
v2_ta = v2_ref['Total Alkalinity'].to_numpy(dtype=float)
v2_ec = v2_ref['Electrical Conductance'].to_numpy(dtype=float)
v2_drp = v2_ref['Dissolved Reactive Phosphorus'].to_numpy(dtype=float)

# 在线已验证 EC 锚点
ec_control_online = v10_best['Electrical Conductance'].to_numpy(dtype=float)

# 距离衰减因子
scale_val = distance_scale(train_feat[['Latitude','Longitude']], val_feat[['Latitude','Longitude']])

# EC 基线模型（用于 guard 分支）
ec_base_val = fit_predict_et(train_feat, val_feat, base_features, 'Electrical Conductance', transform='none', n_estimators=900)

# TA base + TA linked
train_ta_link = train_feat.copy()
val_ta_link = val_feat.copy()
train_ta_link['ec_anchor_feature'] = train_feat['Electrical Conductance'].to_numpy(dtype=float)
val_ta_link['ec_anchor_feature'] = ec_control_online

ta_base = fit_predict_et(train_feat, val_feat, base_features, 'Total Alkalinity', transform='none', n_estimators=900)
ta_link = fit_predict_et(train_ta_link, val_ta_link, base_features + ['ec_anchor_feature'], 'Total Alkalinity', transform='none', n_estimators=900)

# DRP base + DRP linked
train_drp_link = train_feat.copy()
val_drp_link = val_feat.copy()
train_drp_link['ec_anchor_feature'] = train_feat['Electrical Conductance'].to_numpy(dtype=float)
val_drp_link['ec_anchor_feature'] = ec_control_online

drp_base = fit_predict_et(train_feat, val_feat, base_features, 'Dissolved Reactive Phosphorus', transform='log1p', n_estimators=900)
drp_link = fit_predict_et(train_drp_link, val_drp_link, base_features + ['ec_anchor_feature'], 'Dissolved Reactive Phosphorus', transform='log1p', n_estimators=900)

print('components ready.')
print('base_features:', len(base_features))

## Step 10. Render Strategy And Export

**这个步骤做什么**
按本分支参数渲染并导出最终 csv。

**为什么要做这个步骤**
确保每个分支文件可单独复现。

**预期产出**
得到 `v11_4_guard_ec.csv` 文件。

In [ ]:
# =========================
# Step: Render Strategy (v11_4_guard_ec.csv)
# 作用：按本策略参数生成提交文件
# =========================

# ---- 1) EC 分支 ----
ec_mode = 'guard_strong'
if ec_mode == 'control':
    ec = ec_control_online.copy()
elif ec_mode == 'guard':
    ec = ec_tail_guard(ec_control_online, ec_base_val, scale_val, strong=False)
elif ec_mode == 'guard_strong':
    ec = ec_tail_guard(ec_control_online, ec_base_val, scale_val, strong=True)
else:
    raise ValueError(ec_mode)

# ---- 2) TA 分支 ----
# 近点更相信模型，远点更保守
w_ta = 0.45
p_ta = 1.6
near_boost_ta = 0.3
ta = blend_with_scale(v2_ta, ta_link, max_w=w_ta, scale=scale_val, power=p_ta, near_boost=near_boost_ta)

# ---- 3) DRP 分支 ----
w_drp = 0.06
p_drp = 1.3
drp = blend_with_scale(v2_drp, drp_link, max_w=w_drp, scale=scale_val, power=p_drp, near_boost=0.0)

# ---- 4) 导出 ----
desc = f'ec_mode={ec_mode}; w_ta={w_ta}; p_ta={p_ta}; near_boost_ta={near_boost_ta}; w_drp={w_drp}; p_drp={p_drp}'
print('strategy:', desc)

sub = build_submission(val_tpl, ta, ec, drp)
out_file = PROJECT_ROOT / 'v11_4_guard_ec.csv'
sub.to_csv(out_file, index=False)

print('saved:', out_file)
print('shape:', sub.shape)
print('shift_ta_vs_v2:', np.mean(np.abs(sub['Total Alkalinity'] - v2_ref['Total Alkalinity'])))
print('shift_ec_vs_v2:', np.mean(np.abs(sub['Electrical Conductance'] - v2_ref['Electrical Conductance'])))
print('shift_drp_vs_v2:', np.mean(np.abs(sub['Dissolved Reactive Phosphorus'] - v2_ref['Dissolved Reactive Phosphorus'])))

sub.head()

## Step 11. Result Interpretation

**这个步骤做什么**
解释本分支输出和风险特征。

**为什么要做这个步骤**
便于后续 round12 精细化迭代。

**预期产出**
得到可执行的下一轮调整依据。